# Customer Churn Prediction - Model Inference Verification

In [1]:
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model

In [2]:
# Load Model and Preprocessors
model = load_model("model.h5")

with open("label_encoder_gender.pkl", "rb") as f:
    label_encoder_gender = pickle.load(f)

with open("onehot_encoder_geo.pkl", "rb") as f:
    onehot_encoder_geo = pickle.load(f)

with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

In [3]:
# Test Customer Data
customer_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000.0,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000.0
}

# Create DataFrame
input_data = pd.DataFrame([customer_data])

# Encode Gender
input_data["Gender"] = label_encoder_gender.transform(input_data["Gender"])

# One-Hot Encode Geography
geo_encoded = onehot_encoder_geo.transform(input_data[["Geography"]])
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(["Geography"])
)

# Concatenate Encoded Features
input_data_encoded = pd.concat(
    [input_data.drop("Geography", axis=1), geo_encoded_df],
    axis=1
)

# Scale Features
input_scaled = scaler.transform(input_data_encoded)

# Predict Churn Probability
prediction = model.predict(input_scaled, verbose=0)
prediction_proba = float(prediction[0][0])

print(f"Predicted Churn Probability: {prediction_proba:.4f} ({prediction_proba * 100:.2f}%)")